In [6]:

import sys
from utils import *
from Crypto.Util.number import inverse, long_to_bytes

sys.setrecursionlimit(100000000)

def cm_factor(N, D, B=32, debug=True):
    """
    Implements a simplified version of Cheng's 4p - 1 elliptic curve complex multiplication based factorization algorithm.
    Targets moduli where one (or many) prime factors satisfies the equality 4p_i - 1 = D * s^2, where D is a squarefree integer
    and s is a randomly generated one between an upper and lower bound.
    
    Original paper title: I want to break square-free: The 4p - 1 factorization method and its RSA backdoor viability 
    Link: https://crocs.fi.muni.cz/_media/public/papers/2019-secrypt-sedlacek.pdf
    
    :param N: integer to be factored
    :param D: squarefree integer satisfying 4p - 1 = D * s^2
    :param B: number of iterations to run the algorithm for
    :param debug: switches debugging information on/off
    :return: a tuple corresponding to p, q, or failure (-1)
    """

    # If D is not squarefree then we terminate immediately.
    assert D.is_squarefree(), "D must be squarefree."

    # Computes the -Dth Hilbert polynomial modulo N and quotient ring Q = Z_N[x] / <H_{-D, N}>

    Z_N = Zmod(N)
    P = Z_N[x]
    H = P(hilbert_class_polynomial(-D))
    Q = P.quotient_ring(H)

    # j is the equivalence class corresponding to [X] in Q.
    j = Q(x)

    # The paper claims that we can treat both 1728 - j and H as polynomials in Z_N[X] and calculate the inverse of
    # 1728 - j and H via egcd. This doesn't quite work off the shelves, so we instead accomplish this by treating 
    # 1728 - j as an element of Q = Z_N[x] / <H_{-D, n}> and lifting it back into the base ring.  
    # Sage implements this via the .lift() method.

    if debug:
        print("Calculating inverse mod of 1728 - j with H...")

    try:
        k = j * polynomial_inv_mod((1728 - j).lift(), H)
    except ValueError as e:
        r = gcd(int(e.args[1].lc()), N)
        return int(r), int(N // r)

    # Constructs an elliptic curve described by the equation y^2 = x^3 + ax + b where a = 3k and b = 2k over Q.
    # This is so we can calculate the division polynomial psi_n(a, b, x_i) later.
    E = EllipticCurve(Q, [3*k, 2*k])

    for i in range(B):

        # Obtains a random element from Z_N for the division polynomial.
        x_i = Z_N.random_element()

        if debug:
            print("Calculating division polynomial with x_i...")

        # Calculates the division polynomial modulo n: psi_n(a, b, x_i)
        z = E.division_polynomial(N, x=Q(x_i))

        # If our egcd implementation throws an exception, then we know that the r polynomial during the computation
        # has no inverse; we return this polynomial in our exception and then take the gcd of its leading coefficient with N.
        # Otherwise we take the gcd of d and N normally.
        try:
            d, _, _ = polynomial_egcd(z.lift(), H)
        except ValueError as e:
            r = gcd(int(e.args[1].lc()), N)
            return int(r), int(N // r)

        r = gcd(int(d), N)

        # Ensure that the factorization is nontrivial: i.e. r != 1 and r != N
        if 1 < r < N:
            return r, N // r
    
    return -1

if __name__ == "__main__":
    D = 427
    N = 330961752887996173328854935965112588884403584531022561119743740650364958220684640754584393850796812833007843940336784599719224428969119533284286424077547165101460469847980799370419655082069179038497637761333327079374599506574723892143817226751806802676013225188467403274658211563655876500997296917421904614128056847977798146855336939306463059440416150493262973269431000762285579221126342017624118238829230679953011897314722801993750454924627074264353692060002758521401544361385231354313981836056855582929670811259113019012970540824951139489146393182532414878214182086999298397377845534568556100933934481180701997394558264969597606662342898026915506749002491326250792107348176681795942799954526068501499100232598658650184565873243525176833451664254917655703178472944744658628534195346977023418550761620254528178516972066618936960223660362493931786389085393392950207048675797593816271435700130995225483316625836104802608163745376633884840588575355936746173068655319645572100149515524131883813773486917122153248495022372690912572541775943614626733948206252900473118240712831444072243770979419529210034883903111038448366933374841531126421441232024514486168742686297481063089161977054825621099768659097509939405315056325336120929492838479309609958696957890570295444494277819063443427972643459784894450787015151715676537385237767990406742547664321563688829289809321534752244260529319454316532580416182438749849923354060125229328043961355894086576238519138868298499249023773237770103057707912709725417033309061308880583988666463892828633292839968866953776989722310954204550783825704710017434214644199415756584929214239679433211393230307782953067246529626136446314941258877439356094775337541321331600788042698664632064112896956898222397445497695982546922871549828242938368486774617350420790711093069910914135319635330786253331223459637232106417577225350441291
    print("Factoring...")
    r, s = cm_factor(N, D)
    assert r * s == N
    print(f"Factorization successful!")
    print(f"Successfully find r and s, try to solve RSA")
    c = 212346990585867204560749978406244213251144548138396149876252930813225573964743743310623240076921482426544897728053124373315032929324133875314957266495139484699567528919690660711722222757624739519047423652631612556832113937491418819741530730836305579152302101616418756366798409439714043836489654997308261399789368847924248898076302503602653274829239053799622592966934018805688066167076708873399729510077548471187650270072877991644415706656886220388746860133005018914685022732719262919710874193422600082385583629822405629220392922887898181915985227173881016604498106652501844316026711412956745976171977119703328814921559308363060138203377963970984712220593336713159734968921097855310094948675421549640099769461209367374396482468152129218666438977024522653691528013528015440313034981983973685977844334196152779256456715583345981234311241518377392361450912536318491335347506959721906982903993728410239088038142803971132404444984691815547201058733797833802388895579267131452325576042575399419473847790510049088515795639055741434593740667515359201087211978944672386143676684477951410239359787877543382974142784694266278769539348943087451443086676857453247653232890469797970446429750318117491749329986347358826779764098893564081668934172047327871380355612977733499211711098892738572628861337162120664600564875970420208200656165153975734830572375804620405804831699449133408497126664753227381291735116991284383676729731736719032544070913866946805107500261134522561008047413499767639218832012703609787581477514089093664923903581385136180130749154323775598371241866152815888072763101593178296658155000487023879474002640710892103951717343305638528289523819399179422853068955397100518956727546826783306797955126590186360647747653520107951377821594811086096453984438199315753748764774337233049084739849310994774328174885402775747471641470284242744811928033021628049
    e = 65537
    phi = (r-1)*(s-1)
    d = inverse(e, phi)
    m = pow(c, d, N)
    print(long_to_bytes(int(m)))

Factoring...
Calculating inverse mod of 1728 - j with H...
Calculating division polynomial with x_i...
Factorization successful!
Successfully find r and s, try to solve RSA
b'ISCTF{You_have_an_extraordinary_talent_in_cryptography!_But_do_you_understand_number_theory_well_enough?}'
